# Instruct-4DGS x MEGA Hybrid (Colab)

This notebook runs the full Colab + Google Drive workflow:
1. Environment setup and Drive mount
2. Import `cook_spinach` data and pretrained 4DGS outputs
3. Run baseline with three prompts
4. Run hybrid (`color_mode=lite` + entropy)
5. Pack fp16+zip checkpoints and generate reports

In [1]:
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

ROOT = Path('/content/work')
DRIVE_ROOT = Path('/content/drive/MyDrive/EV')
REPO_DIR = ROOT / 'Instruct-4DGS'
DATA_DIR = DRIVE_ROOT / 'data'
OUTPUT_DIR = DRIVE_ROOT / 'output'

ROOT.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('DRIVE_ROOT:', DRIVE_ROOT)

Mounted at /content/drive
ROOT: /content/work
DRIVE_ROOT: /content/drive/MyDrive/EV


In [28]:
%%bash
set -e

cd /content/work
if [ ! -d Instruct-4DGS ]; then
  git clone --recursive https://github.com/juhyeon-kwon/Instruct-4DGS.git
fi

cd /content/work/Instruct-4DGS

python -m pip install -U pip wheel "setuptools<82" jedi

python - <<'PY'
from pathlib import Path
import sys

src = Path('requirements.txt').read_text().splitlines()
out = []
for line in src:
    s = line.strip()
    if not s or s.startswith('#'):
        out.append(line)
        continue
    if s.startswith(('xformers', 'torch==', 'torchvision==', 'torchaudio==', 'transformers==', 'diffusers==', 'accelerate==', 'huggingface-hub==', 'mmcv==')):
        continue
    out.append(s)

out += [
    'transformers==4.46.3',
    'tokenizers==0.20.3',
    'diffusers==0.31.0',
    'accelerate==0.34.2',
    'huggingface-hub==0.26.2',
    'torch>=2.4.0',
    'torchvision>=0.19.0',
    'torchaudio>=2.4.0',
    'mmcv-lite==2.2.0',
    'mmengine==0.10.7',
]

Path('requirements.colab.txt').write_text('\n'.join(out) + '\n')
print('Wrote requirements.colab.txt with', len(out), 'lines')
print('Python:', sys.version)
PY

python -m pip install --prefer-binary -r requirements.colab.txt

# Hard patch source files to avoid direct dependency on mmcv.Config.
python - <<'PY'
from pathlib import Path

targets = ['train.py', 'edit_3d.py', 'refine_sds.py', 'render_edited4d.py']
old = """import mmcv
        from utils.params_utils import merge_hparams
        config = mmcv.Config.fromfile(args.configs)
        args = merge_hparams(args, config)"""
new = """from utils.params_utils import merge_hparams
        try:
            import mmcv
            config = mmcv.Config.fromfile(args.configs)
        except Exception:
            from mmengine.config import Config
            config = Config.fromfile(args.configs)
        args = merge_hparams(args, config)"""

for f in targets:
    p = Path(f)
    if not p.exists():
        print('missing:', f)
        continue
    t = p.read_text()
    if old in t:
        p.write_text(t.replace(old, new))
        print('patched:', f)
    else:
        print('skip:', f)
PY

python - <<'PY'
import mmcv
print('mmcv version:', getattr(mmcv, '__version__', 'unknown'))
print('has Config:', hasattr(mmcv, 'Config'))
print('Fallback patch applied in source files.')
PY

Wrote requirements.colab.txt with 13 lines
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
patched: train.py
patched: edit_3d.py
patched: refine_sds.py
patched: render_edited4d.py
mmcv version: 2.2.0
has Config: False
Fallback patch applied in source files.


In [11]:
%%bash
set -e
cd /content/work

# Build CUDA extensions from 4DGS dependencies.
python -m pip install -U ninja

if [ ! -d 4DGaussians ]; then
  git clone --recursive https://github.com/hustvl/4DGaussians.git
fi

cd /content/work/4DGaussians
git submodule update --init --recursive

# Patch simple-knn for newer CUDA toolchains (FLT_MAX).
python - <<'PY'
from pathlib import Path
p = Path('submodules/simple-knn/simple_knn.cu')
text = p.read_text()
if '#include <cfloat>' not in text and '#include <float.h>' not in text:
    lines = text.splitlines()
    insert_at = 0
    for i, line in enumerate(lines):
        if line.startswith('#include'):
            insert_at = i + 1
    lines.insert(insert_at, '#include <cfloat>')
    p.write_text('\n'.join(lines) + '\n')
    print('Patched simple_knn.cu with <cfloat> include')
else:
    print('simple_knn.cu already has float include')
PY

# Use current torch environment and avoid editable wheel build path.
python -m pip install -v --no-build-isolation submodules/simple-knn
if [ -d submodules/depth-diff-gaussian-rasterization ]; then
  python -m pip install -v --no-build-isolation submodules/depth-diff-gaussian-rasterization
elif [ -d submodules/diff-gaussian-rasterization ]; then
  python -m pip install -v --no-build-isolation submodules/diff-gaussian-rasterization
else
  echo "No rasterization submodule found under 4DGaussians/submodules"
  exit 1
fi

python - <<'PY'
import importlib
mods = ['diff_gaussian_rasterization', 'simple_knn._C']
for m in mods:
    try:
        importlib.import_module(m)
        print(f'[OK] {m}')
    except Exception as e:
        print(f'[FAIL] {m}: {e}')
PY

Patched simple_knn.cu with <cfloat> include
Using pip 26.1.1 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
Processing ./submodules/simple-knn
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for simple_knn: filename=simple_knn-0.0.0-cp312-cp312-linux_x86_64.whl size=3525677 sha256=55fa3839d7951a777620303f7866446a084b4313056c844e32fcbd16dc94a643
  Stored in directory: /root/.cache/pip/wheels/d2/ab/39/3727269be78f69617378c5e0afde42a09a89f9f2ae77da594d
Successfully built simple_knn
Using pip 26.1.1 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
Processing ./submodules/depth-diff-gaussian-rasterization
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for diff_gaussian_rasterization: filename=diff_gaussian_rasterization-0.0.0-cp312-cp312-linux_x86_64.whl size=3801374 sha256=5edb05809b91a8acf6897d876

  Running command Preparing metadata (pyproject.toml)
  running dist_info
  creating /tmp/pip-modern-metadata-4915vb36/simple_knn.egg-info
  writing /tmp/pip-modern-metadata-4915vb36/simple_knn.egg-info/PKG-INFO
  writing dependency_links to /tmp/pip-modern-metadata-4915vb36/simple_knn.egg-info/dependency_links.txt
  writing top-level names to /tmp/pip-modern-metadata-4915vb36/simple_knn.egg-info/top_level.txt
  writing manifest file '/tmp/pip-modern-metadata-4915vb36/simple_knn.egg-info/SOURCES.txt'
  reading manifest file '/tmp/pip-modern-metadata-4915vb36/simple_knn.egg-info/SOURCES.txt'
  writing manifest file '/tmp/pip-modern-metadata-4915vb36/simple_knn.egg-info/SOURCES.txt'
  creating '/tmp/pip-modern-metadata-4915vb36/simple_knn-0.0.0.dist-info'
  Running command Building wheel for simple_knn (pyproject.toml)
  running bdist_wheel
  running build
  running build_ext
  building 'simple_knn._C' extension
  creating /content/work/4DGaussians/submodules/simple-knn/build/temp.linux-

In [3]:
import os
from pathlib import Path

repo = Path('/content/work/Instruct-4DGS')
(repo / 'data').unlink(missing_ok=True) if (repo / 'data').is_symlink() else None
(repo / 'output').unlink(missing_ok=True) if (repo / 'output').is_symlink() else None

if not (repo / 'data').exists():
    os.symlink('/content/drive/MyDrive/EV/data', repo / 'data')
if not (repo / 'output').exists():
    os.symlink('/content/drive/MyDrive/EV/output', repo / 'output')

scene_root = repo / 'data' / 'dynerf' / 'cook_spinach'
ply_init = scene_root / 'points3D_downsample2.ply'
ply_trained = repo / 'output' / 'dynerf' / 'cook_spinach' / 'point_cloud' / 'iteration_14000' / 'point_cloud.ply'
cam_dirs = sorted(scene_root.glob('cam*'))

print('cam folders:', len(cam_dirs))
print(ply_init, '->', 'OK' if ply_init.exists() else 'MISSING')
print(ply_trained, '->', 'OK' if ply_trained.exists() else 'MISSING')

if len(cam_dirs) == 0:
    print('\n[Action needed] camera folders are missing. Put DyNeRF cook_spinach frames under:')
    print(scene_root)
elif not ply_trained.exists():
    print('\n[Action needed] trained point cloud is missing. Run the next training cell once.')
else:
    print('\nReady for editing pipeline.')

cam folders: 21
/content/work/Instruct-4DGS/data/dynerf/cook_spinach/points3D_downsample2.ply -> OK
/content/work/Instruct-4DGS/output/dynerf/cook_spinach/point_cloud/iteration_14000/point_cloud.ply -> MISSING

[Action needed] trained point cloud is missing. Run the next training cell once.


In [30]:
from pathlib import Path
repo = Path("/content/work/Instruct-4DGS")
print("data symlink ->", repo/"data", "=>", (repo/"data").resolve())
print("output symlink ->", repo/"output", "=>", (repo/"output").resolve())

data symlink -> /content/work/Instruct-4DGS/data => /content/drive/MyDrive/EV/data
output symlink -> /content/work/Instruct-4DGS/output => /content/drive/MyDrive/EV/output


In [32]:
from pathlib import Path
import re

p = Path("/content/work/Instruct-4DGS/scene/dataset_readers.py")
t = p.read_text()

pattern = r"""def fetchPly\(path\):\n(?:    .*\n)+?    return BasicPointCloud\(points=positions, colors=colors, normals=normals\)\n"""
replacement = """def fetchPly(path):
    plydata = PlyData.read(path)
    vertices = plydata['vertex']
    positions = np.vstack([vertices['x'], vertices['y'], vertices['z']]).T
    colors = np.vstack([vertices['red'], vertices['green'], vertices['blue']]).T / 255.0
    names = set(vertices.data.dtype.names or [])
    if {'nx', 'ny', 'nz'}.issubset(names):
        normals = np.vstack([vertices['nx'], vertices['ny'], vertices['nz']]).T
    else:
        normals = np.zeros_like(positions, dtype=np.float32)
    return BasicPointCloud(points=positions, colors=colors, normals=normals)
"""

new_t, n = re.subn(pattern, replacement, t, flags=re.MULTILINE)
if n == 0:
    raise RuntimeError("fetchPly block not found; file format may differ.")
p.write_text(new_t)
print("Patched fetchPly in", p)

Patched fetchPly in /content/work/Instruct-4DGS/scene/dataset_readers.py


In [33]:
from pathlib import Path
txt = Path("/content/work/Instruct-4DGS/scene/dataset_readers.py").read_text()
start = txt.find("def fetchPly(path):")
print(txt[start:start+450])

def fetchPly(path):
    plydata = PlyData.read(path)
    vertices = plydata['vertex']
    positions = np.vstack([vertices['x'], vertices['y'], vertices['z']]).T
    colors = np.vstack([vertices['red'], vertices['green'], vertices['blue']]).T / 255.0
    names = set(vertices.data.dtype.names or [])
    if {'nx', 'ny', 'nz'}.issubset(names):
        normals = np.vstack([vertices['nx'], vertices['ny'], vertices['nz']]).T
    else:
        normals = 


In [34]:
%%bash
set -e
cd /content/work/Instruct-4DGS

# Run once to generate ./output/dynerf/cook_spinach/point_cloud/iteration_14000/point_cloud.ply
python train.py \
  --configs ./arguments/dynerf/cook_spinach.py \
  -s ./data/dynerf/cook_spinach \
  --model_path ./output/dynerf/cook_spinach \
  --expname dynerf/cook_spinach

Optimizing ./output/dynerf/cook_spinach
Output folder: ./output/dynerf/cook_spinach [29/05 13:40:49]
feature_dim: 32 [29/05 13:40:49]
meta data loaded, total image:6000 [29/05 13:40:50]
meta data loaded, total image:300 [29/05 13:40:50]
origin points, 3994 [29/05 13:40:50]
after points, 3994 [29/05 13:40:50]
Loading Training Cameras [29/05 13:40:50]
Loading Test Cameras [29/05 13:40:50]
Loading Video Cameras [29/05 13:40:50]
Deformation Net Set aabb [ 22.36910248  14.13998318 181.50617981] [-32.6334343  -78.44248962   0.44451979] [29/05 13:40:50]
Voxel Plane: set aabb= Parameter containing:
tensor([[ 22.3691,  14.1400, 181.5062],
        [-32.6334, -78.4425,   0.4445]]) [29/05 13:40:50]
Number of points at initialisation :  3994 [29/05 13:40:50]
data loading done [29/05 13:40:51]

[ITER 3000] Evaluating test: L1 0.02112022754462326 PSNR 28.455928578096277 [29/05 13:45:31]

[ITER 3000] Evaluating train: L1 0.019784258459420764 PSNR 28.354856042300952 [29/05 13:45:38]
data loading done [

2026-05-29 13:40:46.349625: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-29 13:40:46.420085: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
100%|██████████| 6000/6000 [00:00<00:00, 288493.03it/s]
300it [00:00, 66264.22it/s]
Training progress: 100%|██████████| 14000/14000 [41:10<00:00,  5.67it/s, Loss=0.0093899, psnr=36.23, point=98169] 


In [35]:
PROMPTS = [
    'Make it look like a fauvism painting',
    'Make it look like a sculpture',
    'Turn the man into a woman',
]

DATASET = 'dynerf'
SCENE = 'cook_spinach'
GUIDANCE = 10.5
IMAGE_GUIDANCE = 1.2

print('Prompts:')
for p in PROMPTS:
    print('-', p)

Prompts:
- Make it look like a fauvism painting
- Make it look like a sculpture
- Turn the man into a woman


In [36]:
%%bash
set -e
cd /content/work/Instruct-4DGS

bash script.sh dynerf cook_spinach 10.5 1.2 sh 0.0

bash: script.sh: No such file or directory


CalledProcessError: Command 'b'set -e\ncd /content/work/Instruct-4DGS\n\nbash script.sh dynerf cook_spinach 10.5 1.2 sh 0.0\n'' returned non-zero exit status 127.

In [ ]:
%%bash
set -e
cd /content/work/Instruct-4DGS

bash script.sh dynerf cook_spinach 10.5 1.2 lite 0.002

In [37]:
%%bash
set -e
cd /content/work/Instruct-4DGS

cat > script.sh <<'EOF'
#!/usr/bin/env bash
set -euo pipefail

DATASET="${1:-dynerf}"
SCENE="${2:-cook_spinach}"
GUIDANCE="${3:-10.5}"
IMAGE_GUIDANCE="${4:-1.2}"

TS="$(date +%Y%m%d_%H%M%S)"
LOG_DIR="./output/${DATASET}/${SCENE}/hybrid_logs"
mkdir -p "${LOG_DIR}"
LOG_FILE="${LOG_DIR}/baseline_${TS}.log"

PROMPTS=(
  "Make it look like a fauvism painting"
  "Make it look like a sculpture"
  "Turn the man into a woman"
)

echo "dataset=${DATASET}, scene=${SCENE}" | tee -a "${LOG_FILE}"
for PROMPT in "${PROMPTS[@]}"; do
  echo "======================================================" | tee -a "${LOG_FILE}"
  echo "Running prompt: ${PROMPT}" | tee -a "${LOG_FILE}"
  START="$(date +%s)"

  bash run_instruct_4dgs.sh \
    "${DATASET}" \
    "${SCENE}" \
    "${PROMPT}" \
    "${GUIDANCE}" \
    "${IMAGE_GUIDANCE}" | tee -a "${LOG_FILE}"

  END="$(date +%s)"
  echo "Prompt runtime (sec): $((END - START))" | tee -a "${LOG_FILE}"
done

echo "Done. Log: ${LOG_FILE}" | tee -a "${LOG_FILE}"
EOF

chmod +x script.sh
echo "Wrote /content/work/Instruct-4DGS/script.sh"

Wrote /content/work/Instruct-4DGS/script.sh


In [5]:
%%bash
set -e
cd /content/work/Instruct-4DGS

python - <<'PY'
from pathlib import Path
p = Path("refine_sds.py")
t = p.read_text()

t = t.replace(
"""def encode_vae_in_chunks(ip2p, inputs, batch_size, sample_latent):
    chunks = []
    for i in range(0, inputs.shape[0], batch_size):
        x = inputs[i:i + batch_size]
        with torch.no_grad():
            dist = ip2p.vae.encode(2 * x - 1).latent_dist
            z = dist.sample() if sample_latent else dist.mode()
        chunks.append(z)
    return torch.cat(chunks, dim=0)
""",
"""def encode_vae_in_chunks(ip2p, inputs, batch_size, sample_latent, requires_grad):
    chunks = []
    for i in range(0, inputs.shape[0], batch_size):
        x = inputs[i:i + batch_size]
        if requires_grad:
            dist = ip2p.vae.encode(2 * x - 1).latent_dist
            z = dist.sample() if sample_latent else dist.mode()
        else:
            with torch.no_grad():
                dist = ip2p.vae.encode(2 * x - 1).latent_dist
                z = dist.sample() if sample_latent else dist.mode()
        chunks.append(z)
    return torch.cat(chunks, dim=0)
""")

t = t.replace(
"""        latents = encode_vae_in_chunks(ip2p, vae_input_images, vae_batch_size, sample_latent=True) * 0.18215
        image_latents = encode_vae_in_chunks(ip2p, vae_input_images_cond, vae_batch_size, sample_latent=False)
""",
"""        latents = encode_vae_in_chunks(
            ip2p, vae_input_images, vae_batch_size, sample_latent=True, requires_grad=True
        ) * 0.18215
        image_latents = encode_vae_in_chunks(
            ip2p, vae_input_images_cond, vae_batch_size, sample_latent=False, requires_grad=False
        )
""")

p.write_text(t)
print("[patched] refine_sds.py grad path restored")
PY

[patched] refine_sds.py grad path restored


In [6]:
%%bash
cd /content/work/Instruct-4DGS
bash script.sh

dataset=dynerf, scene=cook_spinach
Running prompt: Make it look like a fauvism painting
------------------------------------------
  - dataset: dynerf
  - scene: cook_spinach
  - prompt: "Make it look like a fauvism painting"
------------------------------------------

[1/4] Collect time0 images...
Found 21 'cam' folders.
✅ Finished copying images to './data/dynerf/time0_cook_spinach'.

[2/4] edit time0 images...
Loaded 21 images from ./data/dynerf/time0_cook_spinach/
✅ Completed time0 image editing.

[3/4] 3D editing
Optimizing ./output/dynerf/cook_spinach
Output folder: ./output/dynerf/cook_spinach [29/05 15:27:39]
feature_dim: 32 [29/05 15:27:39]
meta data loaded, total image:6000 [29/05 15:27:40]
meta data loaded, total image:300 [29/05 15:27:41]
origin points, 3994 [29/05 15:27:41]
after points, 3994 [29/05 15:27:41]
Loading Training Cameras [29/05 15:27:41]
Loading Test Cameras [29/05 15:27:41]
Loading Video Cameras [29/05 15:27:41]
Deformation Net Set aabb [ 22.36910248  14.1399

2026-05-29 15:25:50.744772: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-29 15:25:50.818304: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/lightning_fabric/__init__.py:40: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
INFO:lightning_fabric.utilities.seed:Seed set to 20211202
/usr/local/lib

In [8]:
%%bash
set -e
cd /content/work/Instruct-4DGS

python scripts/pack_model_fp16.py pack \
  --path "./output/dynerf/cook_spinach/point_cloud_refine/Make it look like a fauvism painting/iteration_800"

# Unpack example
# python scripts/pack_model_fp16.py unpack \
#   --path "./output/dynerf/cook_spinach/point_cloud_refine/Make it look like a fauvism painting/iteration_800/packed_fp16.zip" \
#   --output_dir "./output/dynerf/cook_spinach/restore_test/iteration_800"

python3: can't open file '/content/work/Instruct-4DGS/scripts/pack_model_fp16.py': [Errno 2] No such file or directory


CalledProcessError: Command 'b'set -e\ncd /content/work/Instruct-4DGS\n\npython scripts/pack_model_fp16.py pack \\\n  --path "./output/dynerf/cook_spinach/point_cloud_refine/Make it look like a fauvism painting/iteration_800"\n\n# Unpack example\n# python scripts/pack_model_fp16.py unpack \\\n#   --path "./output/dynerf/cook_spinach/point_cloud_refine/Make it look like a fauvism painting/iteration_800/packed_fp16.zip" \\\n#   --output_dir "./output/dynerf/cook_spinach/restore_test/iteration_800"\n'' returned non-zero exit status 2.

In [2]:
import subprocess
from pathlib import Path

log_dir = Path('/content/work/Instruct-4DGS/output/dynerf/cook_spinach/hybrid_logs')
out_md = Path('/content/drive/MyDrive/EV/output/dynerf/cook_spinach/hybrid_logs/ablation_summary.md')

cmd = [
    'python',
    'scripts/summarize_ablation.py',
    '--log_dir', str(log_dir),
    '--output_md', str(out_md),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)
print('Saved:', out_md)

python scripts/summarize_ablation.py --log_dir /content/work/Instruct-4DGS/output/dynerf/cook_spinach/hybrid_logs --output_md /content/drive/MyDrive/EV/output/dynerf/cook_spinach/hybrid_logs/ablation_summary.md


CalledProcessError: Command '['python', 'scripts/summarize_ablation.py', '--log_dir', '/content/work/Instruct-4DGS/output/dynerf/cook_spinach/hybrid_logs', '--output_md', '/content/drive/MyDrive/EV/output/dynerf/cook_spinach/hybrid_logs/ablation_summary.md']' returned non-zero exit status 2.

In [11]:
%%bash
set -e
cd /content/work/Instruct-4DGS

# 1) fauvism
python render_edited4d.py \
  --configs ./arguments/dynerf/cook_spinach.py \
  --ply_path "./output/dynerf/cook_spinach/point_cloud_refine/Make it look like a fauvism painting/iteration_800/point_cloud.ply" \
  -s ./data/dynerf/cook_spinach \
  --model_path ./output/dynerf/cook_spinach
cp "./output/dynerf/cook_spinach/edited_cook_spinach.mp4" "./output/dynerf/cook_spinach/fauvism.mp4"

# 2) sculpture
python render_edited4d.py \
  --configs ./arguments/dynerf/cook_spinach.py \
  --ply_path "./output/dynerf/cook_spinach/point_cloud_refine/Make it look like a sculpture/iteration_800/point_cloud.ply" \
  -s ./data/dynerf/cook_spinach \
  --model_path ./output/dynerf/cook_spinach
cp "./output/dynerf/cook_spinach/edited_cook_spinach.mp4" "./output/dynerf/cook_spinach/sculpture.mp4"

# 3) woman
python render_edited4d.py \
  --configs ./arguments/dynerf/cook_spinach.py \
  --ply_path "./output/dynerf/cook_spinach/point_cloud_refine/Turn the man into a woman/iteration_800/point_cloud.ply" \
  -s ./data/dynerf/cook_spinach \
  --model_path ./output/dynerf/cook_spinach
cp "./output/dynerf/cook_spinach/edited_cook_spinach.mp4" "./output/dynerf/cook_spinach/woman.mp4"

Looking for config file in ./output/dynerf/cook_spinach/cfg_args
Config file found: ./output/dynerf/cook_spinach/cfg_args
Rendering  ./output/dynerf/cook_spinach/point_cloud_refine/Make it look like a fauvism painting/iteration_800/point_cloud.ply
feature_dim: 32 [29/05 17:05:42]
Loading trained model at iteration 14000 [29/05 17:05:42]
meta data loaded, total image:6000 [29/05 17:05:44]
meta data loaded, total image:300 [29/05 17:05:44]
origin points, 3994 [29/05 17:05:44]
after points, 3994 [29/05 17:05:44]
Loading Training Cameras [29/05 17:05:44]
Loading Test Cameras [29/05 17:05:44]
Loading Video Cameras [29/05 17:05:44]
Deformation Net Set aabb [ 22.36910248  14.13998318 181.50617981] [-32.6334343  -78.44248962   0.44451979] [29/05 17:05:44]
Voxel Plane: set aabb= Parameter containing:
tensor([[ 22.3691,  14.1400, 181.5062],
        [-32.6334, -78.4425,   0.4445]]) [29/05 17:05:44]
loading model from exists./output/dynerf/cook_spinach/point_cloud/iteration_14000 [29/05 17:05:44]


100%|██████████| 6000/6000 [00:00<00:00, 262844.92it/s]
300it [00:00, 50661.96it/s]
Rendering progress: 100%|██████████| 300/300 [00:11<00:00, 25.82it/s]
IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1352, 1014) to (1360, 1024) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).
100%|██████████| 6000/6000 [00:00<00:00, 255586.61it/s]
300it [00:00, 64870.40it/s]
Rendering progress: 100%|██████████| 300/300 [00:11<00:00, 26.65it/s]
IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1352, 1014) to (1360, 1024) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).
100%|██████████| 6000/6000 [00:00<00:00, 269

In [12]:
from pathlib import Path
p = Path("/content/work/Instruct-4DGS/output/dynerf/cook_spinach")
for f in sorted(p.glob("*.mp4")):
    print(f)

/content/work/Instruct-4DGS/output/dynerf/cook_spinach/edited_cook_spinach.mp4
/content/work/Instruct-4DGS/output/dynerf/cook_spinach/fauvism.mp4
/content/work/Instruct-4DGS/output/dynerf/cook_spinach/sculpture.mp4
/content/work/Instruct-4DGS/output/dynerf/cook_spinach/woman.mp4


In [ ]:
from pathlib import Path
import pandas as pd

root = Path("/content/work/Instruct-4DGS/output/dynerf/cook_spinach")
prompts = [
    "Make it look like a fauvism painting",
    "Make it look like a sculpture",
    "Turn the man into a woman",
]

def size_mb(p: Path):
    if not p.exists():
        return 0.0
    if p.is_file():
        return p.stat().st_size / (1024**2)
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / (1024**2)

rows = []
for prompt in prompts:
    d3 = root / "point_cloud_3dedit" / prompt / "iteration_1000"
    rf = root / "point_cloud_refine" / prompt / "iteration_800"

    rows.append({
        "prompt": prompt,
        "3dedit_dir_mb": round(size_mb(d3), 2),
        "3dedit_ply_mb": round(size_mb(d3 / "point_cloud.ply"), 2),
        "3dedit_deform_mb": round(size_mb(d3 / "deformation.pth"), 2),
        "refine_dir_mb": round(size_mb(rf), 2),
        "refine_ply_mb": round(size_mb(rf / "point_cloud.ply"), 2),
        "refine_deform_mb": round(size_mb(rf / "deformation.pth"), 2),
        "video_exists_fauvism": (root / "fauvism.mp4").exists(),
        "video_exists_sculpture": (root / "sculpture.mp4").exists(),
        "video_exists_woman": (root / "woman.mp4").exists(),
    })

df_storage = pd.DataFrame(rows)
display(df_storage)

out_csv = root / "storage_summary.csv"
df_storage.to_csv(out_csv, index=False)
print("saved:", out_csv)

In [ ]:
%%bash
set -e
cd /content/work/Instruct-4DGS

LOG_DIR=./output/dynerf/cook_spinach/hybrid_logs
mkdir -p "$LOG_DIR"
STAMP=$(date +%Y%m%d_%H%M%S)
GPU_LOG="$LOG_DIR/gpu_${STAMP}.csv"

# background monitor
nvidia-smi --query-gpu=timestamp,name,memory.total,memory.used,utilization.gpu --format=csv -l 1 > "$GPU_LOG" &
MON_PID=$!

# run one prompt first (example)
bash run_instruct_4dgs.sh dynerf cook_spinach "Make it look like a fauvism painting" 10.5 1.2

# stop monitor
kill $MON_PID || true
echo "gpu log saved: $GPU_LOG"

In [ ]:
import pandas as pd
from pathlib import Path

log_dir = Path("/content/work/Instruct-4DGS/output/dynerf/cook_spinach/hybrid_logs")
log = sorted(log_dir.glob("gpu_*.csv"))[-1]
df = pd.read_csv(log, skipinitialspace=True)

mem_col = [c for c in df.columns if "memory.used" in c][0]
util_col = [c for c in df.columns if "utilization.gpu" in c][0]

df[mem_col] = df[mem_col].str.replace(" MiB","", regex=False).astype(float)
df[util_col] = df[util_col].str.replace(" %","", regex=False).astype(float)

print("log:", log)
print("peak memory (MiB):", df[mem_col].max())
print("avg util (%):", round(df[util_col].mean(), 2))

In [14]:
import pandas as pd
from pathlib import Path

log = sorted(Path("/content/work/Instruct-4DGS/output/dynerf/cook_spinach/hybrid_logs").glob("gpu_*.csv"))[-1]
df = pd.read_csv(log, skipinitialspace=True)
# column name usually like " memory.used [MiB]"
mem_col = [c for c in df.columns if "memory.used" in c][0]
peak = df[mem_col].str.replace(" MiB","", regex=False).astype(float).max()
print("peak GPU memory (MiB):", peak)

IndexError: list index out of range